# Baseline Training — ResNet-20 on CIFAR-10

Vanilla SGD (no momentum) with a step-decay schedule, following the original ResNet paper setup.

In [2]:
import sys
from pathlib import Path
import torch
import torch.nn as nn

sys.path.append(str(Path.cwd().parent))



from src.data import get_loaders
from src.model import resnet20

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
train_loader, test_loader = get_loaders(batch_size=128)

len(train_loader), len(test_loader)

/Users/alexandre/anaconda3/envs/OptML/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


(391, 79)

In [4]:
model = resnet20().to(device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.0,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[82, 123],
    gamma=0.1
)

criterion = nn.CrossEntropyLoss()

In [5]:
from tqdm.notebook import tqdm

num_epochs = 164

for epoch in tqdm(range(num_epochs), desc="Training progress"):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {running_loss/len(train_loader):.4f}")

Training progress:   0%|          | 0/164 [00:00<?, ?it/s]

/Users/alexandre/anaconda3/envs/OptML/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

print(f"Test accuracy: {100 * correct / total:.2f}%")